In [1]:
import requests

session = requests.sessions.Session()

In [2]:
from io import StringIO
import pandas as pd

term = "202601"

main_df = pd.DataFrame()
count = 0
while True:
    response = session.get(f"https://duckweb.uoregon.edu/duckweb/hwskdhnt.P_ListCrse?term_in={term}&sel_subj=dummy&sel_day=dummy&sel_schd=dummy&sel_insm=dummy&sel_camp=dummy&sel_levl=dummy&sel_sess=dummy&sel_instr=dummy&sel_ptrm=dummy&sel_attr=dummy&sel_cred=dummy&sel_tuition=dummy&sel_open=dummy&sel_weekend=dummy&sel_title=&sel_to_cred=&sel_from_cred=&sel_subj=%25&sel_crse=&sel_crn=&begin_hh=0&begin_mi=0&begin_ap=a&end_hh=0&end_mi=0&end_ap=a&submit_btn=Summary&cidx={count}")
    if 'No classes were found that meet your search criteria' in response.text:
        break
    with StringIO(response.text) as buffer:
        df_group = pd.read_html(buffer)

    df = [x for x in df_group if x.iloc[0, 0] == 'Subj'][0].iloc[:-3]
    df.columns = df.iloc[0]
    df = df.loc[:, df.columns.notna()]
    df = df[1:]
    main_df = pd.concat([main_df, df], ignore_index=True)
    count += 100

main_df = main_df.drop_duplicates('CRN').reset_index(drop=True)

def get_parent_crn(row:pd.Series):
    if '+' not in row['Title']:
        return None
    prev_course = main_df.loc[row.name - 1]
    while '+' in prev_course['Title']:
        prev_course = main_df.loc[prev_course.name - 1]
    return prev_course['CRN']


main_df['parent_crn'] = main_df.apply(get_parent_crn, axis=1)
main_df

,Subj,Crse,Title,Creds,CRN,Avail,Max,Time,Day,Location,Instructor,Notes,parent_crn
0,AAAP,410,Cemetery Preservation,4.00 cr.,15837,25,25,1400-1520,tr 9/28-11/05,See DuckWeb,Leifeste A,NaN,None
1,AAAP,411,Intro Historic Preserv,3.00 cr.,10322,20,30,1000-1120,mw,See DuckWeb,Leifeste A,NaN,None
2,AAAP,510,Cemetery Preservation,4.00 cr.,15838,25,25,1400-1520,tr 9/28-11/05,See DuckWeb,Leifeste A,NaN,None
3,AAAP,511,Intro Historic Preserv,3.00 cr.,10323,14,14,1000-1120,mw,See DuckWeb,Leifeste A,NaN,None
4,AAAP,606,Practicum,1.00 cr.,10324,14,14,1000-1150,r,See DuckWeb,Grieger K,8,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4926,WR,320,Scientific & Techn Wr,4.00 cr.,15581,22,22,1000-1120,tr,See DuckWeb,tba,!,None
4927,WR,320,Scientific & Techn Wr,4.00 cr.,15580,22,22,-,NaN,ASYNC WEB,tba,!$,None
4928,WR,321,Business Communication,4.00 cr.,15583,22,22,1200-1320,tr,See DuckWeb,tba,!,None
4929,WR,321,Business Communication,4.00 cr.,15582,22,22,-,NaN,ASYNC WEB,tba,!$8,None


In [3]:
import sqlite3

conn = sqlite3.connect('classes.db')

main_df.to_sql('Classes', conn, index=False, if_exists='replace')
conn.close()